# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source

The dataset is described by a Croissant schema and references record sets and fields using their `@id` identifiers. This ensures unambiguous and reproducible access to each part of the dataset.


In [ ]:
# Make sure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading

Let's load the dataset metadata and inspect its description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL for this dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Initialize Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Date Published: {metadata.datePublished}")
print(f"License: {metadata.license}\n")
if hasattr(metadata, "keywords"):
    print("Keywords:", ", ".join(metadata.keywords))

## 2. Data Overview

Let's review the available record sets and their `@id`s. In Croissant, a record set defines a table-like structure for tabular data, and each field or column also has its unique `@id`.

First, enumerate all record sets and their contained fields and columns.

In [ ]:
record_sets = list(dataset.record_sets())
if not record_sets:
    print("This dataset does not define record sets directly in the package metadata (for privacy or schema brevity). Let's check if the dataset loads any from available distribution.")
else:
    print(f"Number of record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        print(f"  Name: {rs.get('name', '[no name]')}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
                print(f"    - Field @id: {fid}")
        print()

# If record sets are not available from metadata, try to retrieve their IDs via the dataset implementation
if not record_sets:
    available_record_sets = dataset.available_record_sets()
    print("Available record sets (by @id):")
    for rs in available_record_sets:
        print(f"- {rs}")

## 3. Data Extraction

We'll extract data from the available record sets. Since the Croissant schema may provide record sets dynamically, we'll use their `@id` as required by the specification. We'll load records from each record set into a DataFrame for further analysis.


In [ ]:
# Discover record set IDs dynamically
record_set_ids = dataset.available_record_sets()
print("Available record set @ids:", record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nLoading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame shape for {record_set_id}: {df.shape}")
        print("Columns:", list(df.columns))
    else:
        print(f"No records found for record set {record_set_id}.")

# For consistent reference in later cells, select the first (main) record set
main_record_set_id = record_set_ids[0] if record_set_ids else None

# Show a preview if available
if main_record_set_id and main_record_set_id in dataframes:
    print(f"\nPreview of DataFrame for RecordSet @id: {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Let's process and explore the data. Here, we'll:
- Select a numeric field (by column `@id`) for filtering and normalization.
- Remove outliers (simple threshold), normalize the values, and group by a categorical attribute if one exists.

Make sure to use the `@id` of columns in all references.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Columns in main DataFrame ({main_record_set_id}):\n", list(df.columns))

    # Try to find a numeric field
    numeric_field_id = None
    for col in df.columns:
        # We'll guess types crudely here; ideally, metadata fields would be available
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        print(f"\nUsing numeric field for analysis: {numeric_field_id}")
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > mean ({threshold:.2f}): {len(filtered_df)} records")

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized field '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to find a categorical/group field (non-numeric, not the index)
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and (
                df[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df[col])
            ):
                group_field_id = col
                break

        if group_field_id:
            print(f"\nGrouping filtered records by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric field found in the main record set.")
else:
    print("Main record set DataFrame unavailable for EDA.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field, and (if available) compare its normalized values across groupings.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(12, 6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f'{numeric_field_id} by {group_field_id} (Filtered, > mean)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization skipped: requirements missing (numeric field or DataFrame not loaded).")

## 6. Conclusion

In this notebook, we explored the FAIR^2 dataset defined by a Croissant schema. We:
- Loaded metadata and data from the Croissant schema using the `mlcroissant` library.
- Discovered record sets and identified fields by `@id`.
- Extracted data into DataFrames and performed basic exploratory data analysis, including filtering and normalization by numeric fields, and grouping by categorical attributes where identified.
- Visualized distributions and group differences with Matplotlib and Seaborn.

**Note:** For reproducible analysis, always use entity `@id`s rather than display names or column headings, as Croissant's design ensures these are stable and unambiguous.

This workflow can be adapted for any Croissant-based dataset using the same approach. For further learning, see the [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/python).
